# Narração com VoxCPM2 (Google Colab)

Gera a narração de um projeto do video-coding com o **VoxCPM2** (openbmb, licença Apache 2.0, uso comercial liberado).

1. No PC: `python tools/exportar_falas.py PASTA_DO_PROJETO` grava `falas.json`.
2. Aqui: menu **Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4**.
3. Ajuste a célula **Configuração** e use **Ambiente de execução > Executar tudo**. A célula de envio pede o
   `falas.json` (e o áudio de referência, se for clonar).
4. No fim baixa `falas.zip`. No PC:
   - uma versão por fala: `python tools/import_voice.py PASTA_DO_PROJETO falas.zip pt`
   - várias versões (`VERSOES > 1`): extraia o zip e rode `python tools/escolher_consistente.py PASTA_DO_PROJETO PASTA_EXTRAIDA`

Velocidade numa T4: cerca de 2 s de GPU por 1 s de áudio, mais uns 3 min para carregar o modelo.
**Clonar a voz de uma pessoa exige a autorização dela.**


In [ ]:
!nvidia-smi -L
!pip install -q voxcpm soundfile


## Configuração

In [ ]:
# Desenho de voz: descrição entre parênteses (vazio = voz padrão do modelo). Com referência, vira "clone com estilo".
ESTILO = '(Homem brasileiro de uns trinta anos, voz média e clara, fala articulada, tom didático)'
CLONAR = False        # True: envie um áudio de referência (10 a 30 s de fala limpa) na próxima célula
TEXTO_DA_REFERENCIA = ''   # o que é dito no áudio de referência (deixa o clone mais fiel); vazio = só o timbre
VERSOES = 1           # versões de cada fala (2 a 4 para escolher a mais uniforme com escolher_consistente.py)
CFG = 2.0             # mais alto = mais fiel e marcado; 1,5 = mais solto
PASSOS = 10           # passos de refinamento (15 a 20: um pouco mais limpo e mais lento)


## Envio dos arquivos

In [ ]:
import json
from google.colab import files
up = files.upload()   # escolha falas.json (e o áudio de referência, se CLONAR)
FALAS = json.loads(up['falas.json'].decode('utf-8'))
REF = next((k for k in up if k != 'falas.json'), None) if CLONAR else None
print(len(FALAS['falas']), 'falas; referência:', REF)


## Geração

In [ ]:
import time, zipfile, numpy as np, soundfile as sf
from pathlib import Path
from voxcpm import VoxCPM

t0 = time.time(); model = VoxCPM.from_pretrained('openbmb/VoxCPM2', load_denoiser=False); sr = model.tts_model.sample_rate
print(f'modelo carregado em {time.time()-t0:.0f} s, {sr} Hz')
OUT = Path('falas'); OUT.mkdir(exist_ok=True)
extra = {}
if REF:
    extra['reference_wav_path'] = REF
    if TEXTO_DA_REFERENCIA: extra.update(prompt_wav_path=REF, prompt_text=TEXTO_DA_REFERENCIA)
for f in FALAS['falas']:
    for k in range(VERSOES):
        t0 = time.time()
        wav = np.asarray(model.generate(text=ESTILO + f['texto'], cfg_value=CFG, inference_timesteps=PASSOS,
                                        normalize=True, retry_badcase=True, **extra), dtype=np.float32)
        nome = f"{f['id']}.wav" if VERSOES == 1 else f"{f['id']}_v{k}.wav"
        sf.write(OUT/nome, wav, sr)
        print(f'{nome}: {len(wav)/sr:.1f} s em {time.time()-t0:.0f} s')
with zipfile.ZipFile('falas.zip', 'w') as z:
    for p in sorted(OUT.glob('*.wav')): z.write(p, p.name)
files.download('falas.zip')
